# Step 09 - Analytics

## Mục tiêu

- Phân tích dữ liệu từ Gold Layer.
- Tính các KPI chính của hệ thống E-commerce.
- Phân tích doanh thu theo thời gian.
- Xác định category và sản phẩm có doanh thu cao.
- Phân tích hành vi khách hàng.
- Phân tích đánh giá của khách hàng.
- Phân tích thanh toán và chênh lệch giữa giá trị đơn hàng và payment.
- Chuẩn bị dữ liệu phục vụ Dashboard ở bước tiếp theo.

In [2]:
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ICEBERG_VERSION = "1.11.0"
HADOOP_VERSION = "3.4.2"

PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:{ICEBERG_VERSION},"
    f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
    f"org.apache.hadoop:hadoop-aws:{HADOOP_VERSION}"
)

AWS_ACCESS_KEY_ID = (
    os.environ.get("AWS_ACCESS_KEY_ID")
    or os.environ.get("MINIO_ROOT_USER")
)

AWS_SECRET_ACCESS_KEY = (
    os.environ.get("AWS_SECRET_ACCESS_KEY")
    or os.environ.get("MINIO_ROOT_PASSWORD")
)

spark = (
    SparkSession.builder
    .appName("EcommerceAnalytics")
    .master("local[*]")

    .config("spark.jars.packages", PACKAGES)

    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    .config(
        "spark.sql.catalog.lakehouse",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.lakehouse.type",
        "rest"
    )
    .config(
        "spark.sql.catalog.lakehouse.uri",
        "http://iceberg-rest:8181"
    )

    .config(
        "spark.sql.catalog.lakehouse.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.path-style-access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.endpoint.region",
        "us-east-1"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        AWS_ACCESS_KEY_ID
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        AWS_SECRET_ACCESS_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.1.3


In [3]:
spark.sql("""
SHOW TABLES IN lakehouse.gold
""").show(truncate=False)

+---------+-----------------------+-----------+
|namespace|tableName              |isTemporary|
+---------+-----------------------+-----------+
|gold     |category_sales         |false      |
|gold     |customer_summary       |false      |
|gold     |daily_sales            |false      |
|gold     |iceberg_demo           |false      |
|gold     |order_financial_summary|false      |
|gold     |payment_summary        |false      |
|gold     |product_performance    |false      |
|gold     |review_summary         |false      |
|gold     |sales_detail           |false      |
|gold     |state_sales            |false      |
+---------+-----------------------+-----------+



In [4]:
sales_detail = spark.table(
    "lakehouse.gold.sales_detail"
)

daily_sales = spark.table(
    "lakehouse.gold.daily_sales"
)

category_sales = spark.table(
    "lakehouse.gold.category_sales"
)

product_performance = spark.table(
    "lakehouse.gold.product_performance"
)

customer_summary = spark.table(
    "lakehouse.gold.customer_summary"
)

review_summary = spark.table(
    "lakehouse.gold.review_summary"
)

state_sales = spark.table(
    "lakehouse.gold.state_sales"
)

payment_summary = spark.table(
    "lakehouse.gold.payment_summary"
)

order_financial_summary = spark.table(
    "lakehouse.gold.order_financial_summary"
)

In [5]:
kpi_overview = (
    sales_detail
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items"),
        F.countDistinct("customer_id").alias("total_customers"),
        F.sum("item_total").alias("total_revenue"),
        F.avg("item_total").alias("avg_item_value")
    )
)

kpi_overview.show(truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

+------------+-----------+---------------+--------------------+-----------------+
|total_orders|total_items|total_customers|total_revenue       |avg_item_value   |
+------------+-----------+---------------+--------------------+-----------------+
|98666       |112650     |98666          |1.5843553239998858E7|140.6440589436206|
+------------+-----------+---------------+--------------------+-----------------+



## 9.5. Doanh thu theo ngày

Sử dụng bảng `daily_sales` đã được tổng hợp ở Gold Layer để xem xu hướng doanh thu và số lượng đơn hàng theo ngày.


In [ ]:
daily_analysis = (
    daily_sales
    .select(
        "order_date",
        "total_orders",
        "total_items",
        F.round("product_revenue", 2).alias("product_revenue"),
        F.round("freight_revenue", 2).alias("freight_revenue"),
        F.round("total_revenue", 2).alias("total_revenue")
    )
    .orderBy("order_date")
)

daily_analysis.show(20, truncate=False)


## 9.6. Doanh thu theo tháng

Dữ liệu theo tháng dễ quan sát xu hướng hơn so với dữ liệu theo ngày.


In [ ]:
monthly_sales = (
    sales_detail
    .withColumn(
        "year_month",
        F.date_format("order_date", "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items"),
        F.round(F.sum("price"), 2).alias("product_revenue"),
        F.round(F.sum("freight_value"), 2).alias("freight_revenue"),
        F.round(F.sum("item_total"), 2).alias("total_revenue")
    )
    .orderBy("year_month")
)

monthly_sales.show(50, truncate=False)


## 9.7. Top 10 category theo doanh thu


In [ ]:
top_categories = (
    category_sales
    .filter(F.col("product_category_name").isNotNull())
    .select(
        "product_category_name",
        "total_orders",
        "items_sold",
        F.round("product_revenue", 2).alias("product_revenue"),
        F.round("total_revenue", 2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
    .limit(10)
)

top_categories.show(truncate=False)


## 9.8. Top 10 sản phẩm theo doanh thu


In [ ]:
top_products_revenue = (
    product_performance
    .select(
        "product_id",
        "product_category_name",
        "total_orders",
        "items_sold",
        F.round("product_revenue", 2).alias("product_revenue"),
        F.round("total_revenue", 2).alias("total_revenue")
    )
    .orderBy(F.col("product_revenue").desc())
    .limit(10)
)

top_products_revenue.show(truncate=False)


## 9.9. Top 10 sản phẩm theo số lượng bán

Sản phẩm có doanh thu cao nhất chưa chắc là sản phẩm bán được nhiều nhất.


In [ ]:
top_products_quantity = (
    product_performance
    .select(
        "product_id",
        "product_category_name",
        "items_sold",
        F.round("product_revenue", 2).alias("product_revenue")
    )
    .orderBy(
        F.col("items_sold").desc(),
        F.col("product_revenue").desc()
    )
    .limit(10)
)

top_products_quantity.show(truncate=False)


## 9.10. Top 10 khách hàng theo tổng chi tiêu


In [ ]:
top_customers = (
    customer_summary
    .select(
        "customer_id",
        "customer_city",
        "customer_state",
        "total_orders",
        "total_items",
        F.round("total_spent", 2).alias("total_spent"),
        "first_order_date",
        "last_order_date"
    )
    .orderBy(F.col("total_spent").desc())
    .limit(10)
)

top_customers.show(truncate=False)


## 9.11. Phân tích doanh thu theo bang


In [ ]:
state_analysis = (
    state_sales
    .filter(F.col("customer_state").isNotNull())
    .select(
        "customer_state",
        "total_orders",
        "total_items",
        F.round("total_revenue", 2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

state_analysis.show(truncate=False)


## 9.12. Phân tích đánh giá khách hàng


In [ ]:
review_analysis = (
    review_summary
    .select(
        "review_score",
        "total_reviews",
        "percentage"
    )
    .orderBy("review_score")
)

review_analysis.show(truncate=False)


### Điểm review trung bình


In [ ]:
avg_review = (
    spark.table("lakehouse.silver.reviews")
    .agg(
        F.round(
            F.avg("review_score"),
            2
        ).alias("avg_review_score")
    )
)

avg_review.show(truncate=False)


## 9.13. Phân tích phương thức thanh toán

Phần này đọc trực tiếp từ Silver vì bảng `payment_summary` ở Gold đã tổng hợp theo `order_id` và không còn giữ chi tiết `payment_type`.


In [ ]:
payments_silver = spark.table(
    "lakehouse.silver.payments"
)

payment_type_analysis = (
    payments_silver
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("payment_records"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(
            F.sum("payment_value"),
            2
        ).alias("total_payment_value")
    )
    .orderBy(
        F.col("total_payment_value").desc()
    )
)

payment_type_analysis.show(truncate=False)


## 9.14. Phân tích số kỳ trả góp


In [ ]:
installment_analysis = (
    payments_silver
    .groupBy("payment_installments")
    .agg(
        F.count("*").alias("payment_count"),
        F.round(
            F.avg("payment_value"),
            2
        ).alias("avg_payment_value"),
        F.round(
            F.sum("payment_value"),
            2
        ).alias("total_payment_value")
    )
    .orderBy("payment_installments")
)

installment_analysis.show(30, truncate=False)


## 9.15. Phân tích trạng thái đơn hàng


In [ ]:
orders_silver = spark.table(
    "lakehouse.silver.orders"
)

total_order_count = orders_silver.count()

order_status_analysis = (
    orders_silver
    .groupBy("order_status")
    .agg(
        F.count("*").alias("total_orders")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("total_orders")
            / F.lit(total_order_count)
            * 100,
            2
        )
    )
    .orderBy(F.col("total_orders").desc())
)

order_status_analysis.show(truncate=False)


## 9.16. Average Order Value (AOV)

`avg_item_value` là giá trị trung bình của một item.  
AOV phải được tính trên tổng giá trị của từng đơn hàng.


In [ ]:
order_value = (
    sales_detail
    .groupBy("order_id")
    .agg(
        F.sum("item_total").alias("order_value")
    )
)

avg_order_value = (
    order_value
    .agg(
        F.round(
            F.avg("order_value"),
            2
        ).alias("avg_order_value")
    )
)

avg_order_value.show(truncate=False)


## 9.17. So sánh Sales và Payment

Phân tích các đơn có chênh lệch đáng kể giữa tổng giá trị item + freight và tổng payment.


In [ ]:
financial_difference = (
    order_financial_summary
    .filter(
        F.col("payment_difference").isNotNull()
        & (F.abs(F.col("payment_difference")) > 0.01)
    )
    .select(
        "order_id",
        F.round("sales_total", 2).alias("sales_total"),
        F.round("payment_total", 2).alias("payment_total"),
        "payment_count",
        "max_installments",
        F.round(
            "payment_difference",
            2
        ).alias("payment_difference")
    )
    .orderBy(
        F.abs(F.col("payment_difference")).desc()
    )
)

print(
    "Orders with payment difference > 0.01:",
    financial_difference.count()
)

financial_difference.show(
    20,
    truncate=False
)


## 9.18. Tạo bộ KPI cho Dashboard

Các KPI chính:
- Tổng số đơn hàng
- Tổng số khách hàng
- Tổng số sản phẩm
- Tổng doanh thu
- Average Order Value
- Điểm review trung bình


In [ ]:
total_orders = sales_detail.select("order_id").distinct().count()
total_customers = sales_detail.select("customer_id").distinct().count()
total_products = sales_detail.select("product_id").distinct().count()

total_revenue = (
    sales_detail
    .agg(F.sum("item_total").alias("value"))
    .first()["value"]
)

aov = (
    order_value
    .agg(F.avg("order_value").alias("value"))
    .first()["value"]
)

avg_review_score = (
    spark.table("lakehouse.silver.reviews")
    .agg(F.avg("review_score").alias("value"))
    .first()["value"]
)

dashboard_kpi = spark.createDataFrame(
    [(
        total_orders,
        total_customers,
        total_products,
        round(float(total_revenue or 0), 2),
        round(float(aov or 0), 2),
        round(float(avg_review_score or 0), 2)
    )],
    [
        "total_orders",
        "total_customers",
        "total_products",
        "total_revenue",
        "avg_order_value",
        "avg_review_score"
    ]
).withColumn(
    "_generated_at",
    F.current_timestamp()
)

dashboard_kpi.show(truncate=False)


## 9.19. Lưu các bảng Analytics dùng cho Dashboard

Chỉ lưu các kết quả sẽ được tái sử dụng thường xuyên ở Step 10.


In [ ]:
(
    dashboard_kpi
    .writeTo("lakehouse.gold.dashboard_kpi")
    .using("iceberg")
    .createOrReplace()
)

(
    monthly_sales
    .withColumn(
        "_generated_at",
        F.current_timestamp()
    )
    .writeTo("lakehouse.gold.monthly_sales")
    .using("iceberg")
    .createOrReplace()
)

print("✅ gold.dashboard_kpi")
print("✅ gold.monthly_sales")


## 9.20. Kiểm tra các bảng Gold sau Analytics


In [ ]:
spark.sql("""
SHOW TABLES IN lakehouse.gold
""").show(truncate=False)


## 9.21. Kiểm tra dữ liệu chuẩn bị cho Dashboard


In [ ]:
print("===== DASHBOARD KPI =====")
spark.table(
    "lakehouse.gold.dashboard_kpi"
).show(truncate=False)

print("===== MONTHLY SALES =====")
spark.table(
    "lakehouse.gold.monthly_sales"
).orderBy(
    "year_month"
).show(20, truncate=False)

print("===== TOP CATEGORIES =====")
top_categories.show(10, truncate=False)

print("===== REVIEW ANALYSIS =====")
review_analysis.show(truncate=False)


## Kết luận

Trong Step 09, hệ thống đã sử dụng dữ liệu Gold để thực hiện các phân tích nghiệp vụ chính:

- KPI tổng quan của hệ thống E-commerce.
- Doanh thu theo ngày và theo tháng.
- Top category và top sản phẩm.
- Giá trị và hành vi khách hàng.
- Doanh thu theo khu vực.
- Phân bố review score.
- Phương thức thanh toán và trả góp.
- Trạng thái đơn hàng.
- Average Order Value.
- Chênh lệch giữa sales và payment.

Hai bảng `gold.dashboard_kpi` và `gold.monthly_sales` được tạo thêm để hỗ trợ trực tiếp cho Dashboard ở Step 10.
